In [1]:
import pandas as pd
import os

# Dossier racine contenant les CSV
racine = r"C:\Projet-IESE5-Localisation\Résultat mesures"

# Dictionnaire pour stocker les 100 mesures centrales de chaque fichier
mesures = {}

# Parcourir récursivement tous les fichiers CSV
for dirpath, dirnames, filenames in os.walk(racine):
    for filename in filenames:
        if filename.endswith(".csv"):
            chemin = os.path.join(dirpath, filename)
            # Chemin relatif pour identifier le fichier
            cle = os.path.relpath(chemin, racine)
            
            df = pd.read_csv(chemin)
            n = len(df)
            
            # Extraire 100 mesures au milieu
            debut = (n - 100) // 2
            fin = debut + 100
            df_milieu = df.iloc[debut:fin]
            
            # Stocker la colonne "Distance (m)"
            mesures[cle] = df_milieu["Distance (m)"].reset_index(drop=True)
            print(f"{cle}: {n} lignes → extrait lignes {debut} à {fin-1}")

# Créer un DataFrame récapitulatif avec toutes les mesures
df_mesures = pd.DataFrame(mesures)
print(f"\nDataFrame final : {df_mesures.shape}")
df_mesures.head(10)

intérieur\double side\3m perturbé avec wifi.csv: 201 lignes → extrait lignes 50 à 149
intérieur\double side\Obstacle humain\1m avec humain.csv: 122 lignes → extrait lignes 11 à 110
intérieur\double side\Obstacle humain\3m avec humain qui bouge pas.csv: 147 lignes → extrait lignes 23 à 122
intérieur\double side\Obstacle humain\3m avec humain.csv: 158 lignes → extrait lignes 29 à 128
intérieur\double side\Obstacle humain\5m avec humain.csv: 129 lignes → extrait lignes 14 à 113
intérieur\double side\Obstacle humain\9m avec humain.csv: 139 lignes → extrait lignes 19 à 118
intérieur\double side\Obstacle métal\1m avec métal.csv: 134 lignes → extrait lignes 17 à 116
intérieur\double side\Obstacle métal\3m avec métal.csv: 159 lignes → extrait lignes 29 à 128
intérieur\double side\Obstacle métal\5m avec métal.csv: 130 lignes → extrait lignes 15 à 114
intérieur\double side\Obstacle métal\9m avec métal.csv: 129 lignes → extrait lignes 14 à 113
intérieur\double side\Sans obstacles\1m sans obstacle

,intérieur\double side\3m perturbé avec wifi.csv,intérieur\double side\Obstacle humain\1m avec humain.csv,intérieur\double side\Obstacle humain\3m avec humain qui bouge pas.csv,intérieur\double side\Obstacle humain\3m avec humain.csv,intérieur\double side\Obstacle humain\5m avec humain.csv,intérieur\double side\Obstacle humain\9m avec humain.csv,intérieur\double side\Obstacle métal\1m avec métal.csv,intérieur\double side\Obstacle métal\3m avec métal.csv,intérieur\double side\Obstacle métal\5m avec métal.csv,intérieur\double side\Obstacle métal\9m avec métal.csv,intérieur\double side\Sans obstacles\1m sans obstacles.csv,intérieur\double side\Sans obstacles\3m sans obsctacles.csv,intérieur\double side\Sans obstacles\5m sans obstacles.csv,intérieur\double side\Sans obstacles\9m sans obstacles.csv,intérieur\single side\1m single side sans obstacles.csv,intérieur\single side\3m ss sans obstacles.csv,intérieur\single side\5m ss sans obsctacles.csv,intérieur\single side\9m ss sans obstacles.csv
0,3.08,2.14,3.65,3.20,5.15,9.28,2.28,3.27,5.22,9.12,0.87,3.04,5.10,8.89,0.93,3.05,5.10,9.17
1,3.05,1.81,3.84,3.21,5.14,9.31,1.62,3.30,5.20,9.08,0.87,3.06,5.09,8.91,0.93,3.03,5.08,9.19
2,3.06,1.47,3.92,3.19,5.14,9.22,1.84,3.29,5.20,9.09,0.92,3.04,5.10,8.88,0.95,3.03,5.06,9.15
3,3.07,1.47,3.93,3.20,5.15,9.25,2.32,3.30,5.23,9.10,0.89,3.06,5.09,8.90,0.93,3.05,5.00,9.09
4,3.08,2.18,3.89,3.24,5.15,9.22,2.30,3.31,5.23,9.08,0.90,3.05,5.08,8.90,0.91,3.05,5.06,9.11
5,3.09,2.05,3.92,3.17,5.16,9.24,2.05,3.27,5.25,9.09,0.88,3.05,5.11,8.90,0.91,3.07,5.00,9.13
6,3.09,1.68,3.83,3.20,5.15,9.23,2.06,3.25,5.22,9.09,0.87,3.04,5.06,8.91,0.97,3.07,5.06,9.17
7,3.07,1.63,3.78,3.20,5.16,9.25,2.32,3.26,5.25,9.11,0.90,3.06,5.08,8.92,0.99,3.07,5.04,9.17
8,3.08,1.58,3.86,3.18,5.18,9.24,1.83,3.31,5.23,9.08,0.89,3.04,5.03,9.01,0.86,3.07,5.08,9.11
9,3.07,2.18,3.87,3.19,5.16,9.23,1.88,3.36,5.23,9.14,0.89,3.04,5.11,8.99,0.88,3.03,4.97,9.13


In [2]:
import numpy as np
import re

# Extraire la vraie distance depuis le nom du fichier (ex: "1m sans obstacles.csv" → 1.0)
def extraire_distance_vraie(nom_fichier):
    match = re.search(r'(\d+)\s*m', nom_fichier.split('\\')[-1])
    if match:
        return float(match.group(1))
    return None

# Seuil pour valeurs fausses : 5% de la vraie distance
seuil_pct = 5.0

resultats = []

for col in df_mesures.columns:
    valeurs = df_mesures[col].dropna().values
    dist_vraie = extraire_distance_vraie(col)
    
    if dist_vraie is None:
        print(f"⚠ Impossible d'extraire la distance vraie pour : {col}")
        continue
    
    erreurs = valeurs - dist_vraie
    
    biais = np.mean(erreurs)
    rmse = np.sqrt(np.mean(erreurs ** 2))
    ecart_type = np.std(valeurs)
    
    seuil = dist_vraie * (seuil_pct / 100.0)
    pct_fausses = np.sum(np.abs(erreurs) > seuil) / len(valeurs) * 100.0
    
    resultats.append({
        "Fichier": col,
        "Distance vraie (m)": dist_vraie,
        "Biais (m)": round(biais, 4),
        "RMSE (m)": round(rmse, 4),
        "Écart-type (m)": round(ecart_type, 4),
        "Valeurs fausses (%)": round(pct_fausses, 1)
    })

df_stats = pd.DataFrame(resultats)
df_stats.style.format({
    "Biais (m)": "{:+.4f}",
    "RMSE (m)": "{:.4f}",
    "Écart-type (m)": "{:.4f}",
    "Valeurs fausses (%)": "{:.1f}%"
}).background_gradient(subset=["RMSE (m)"], cmap="Reds")\
 .background_gradient(subset=["Valeurs fausses (%)"], cmap="Reds")

,Fichier,Distance vraie (m),Biais (m),RMSE (m),Écart-type (m),Valeurs fausses (%)
0,intérieur\double side\3m perturbé avec wifi.csv,3.000000,+0.0687,0.0704,0.0155,0.0%
1,intérieur\double side\Obstacle humain\1m avec humain.csv,1.000000,+0.1604,0.3397,0.2995,64.0%
2,intérieur\double side\Obstacle humain\3m avec humain qui bouge pas.csv,3.000000,+0.6911,0.6996,0.1086,100.0%
3,intérieur\double side\Obstacle humain\3m avec humain.csv,3.000000,+0.2256,0.2882,0.1793,64.0%
4,intérieur\double side\Obstacle humain\5m avec humain.csv,5.000000,+0.1468,0.1488,0.0245,0.0%
5,intérieur\double side\Obstacle humain\9m avec humain.csv,9.000000,+0.1904,0.2082,0.0841,1.0%
6,intérieur\double side\Obstacle métal\1m avec métal.csv,1.000000,+0.7611,0.8305,0.3324,100.0%
7,intérieur\double side\Obstacle métal\3m avec métal.csv,3.000000,+0.2665,0.2680,0.0285,100.0%
8,intérieur\double side\Obstacle métal\5m avec métal.csv,5.000000,+0.2027,0.2044,0.0265,2.0%
9,intérieur\double side\Obstacle métal\9m avec métal.csv,9.000000,+0.1079,0.1106,0.0245,0.0%
